In [20]:
import base64
import sys
from datetime import datetime
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sqlalchemy import create_engine

# Allow imports from project root when running in Jupyter
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists() and (PROJECT_ROOT.parent / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import REPORT_OUTPUT_DIR, get_sqlalchemy_uri

output_dir = PROJECT_ROOT / REPORT_OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

engine = create_engine(get_sqlalchemy_uri())

# 2. Extract Data
df_kpi = pd.read_sql("SELECT * FROM v_kpi_summary", engine)
df_dist = pd.read_sql("SELECT * FROM v_rating_distribution", engine)
df_metrics = pd.read_sql(
    "SELECT * FROM v_movie_metrics WHERE review_count > 5 ORDER BY avg_rating DESC LIMIT 10",
    engine,
)
df_popular = pd.read_sql(
    "SELECT * FROM v_movie_metrics ORDER BY review_count DESC LIMIT 10", engine
)
df_upvotes = pd.read_sql(
    "SELECT * FROM v_movie_metrics ORDER BY total_upvotes DESC LIMIT 10", engine
)
df_reviewers = pd.read_sql(
    "SELECT * FROM v_top_reviewers ORDER BY total_reviews DESC LIMIT 10", engine
)
df_scatter = pd.read_sql("SELECT * FROM v_movie_metrics", engine)

# 3. Dynamic Chart Values Calculation
top_rating = (
    df_dist.loc[df_dist["review_count"].idxmax(), "rating"]
    if not df_dist.empty
    else "N/A"
)
peak_review_count = (
    f"{df_dist['review_count'].max():,}"
    if not df_dist.empty
    else "N/A"
)
top_reviewer = (
    df_reviewers.iloc[0]["username"] if not df_reviewers.empty else "N/A"
)
top_movie_rating = (
    df_metrics.iloc[0]["title"] if not df_metrics.empty else "N/A"
)
top_movie_vol = df_popular.iloc[0]["title"] if not df_popular.empty else "N/A"
top_movie_upvotes = (
    df_upvotes.iloc[0]["title"] if not df_upvotes.empty else "N/A"
)


# Helper function to generate base64 string for an individual plot
def generate_chart_base64(plot_func, figsize=(8, 5)):
  sns.set_theme(style="whitegrid")
  fig, ax = plt.subplots(figsize=figsize)
  plot_func(ax)
  plt.tight_layout()
  buffer = BytesIO()
  plt.savefig(buffer, format="png", bbox_inches="tight")
  buffer.seek(0)
  img_b64 = base64.b64encode(buffer.read()).decode("utf-8")
  plt.close(fig)
  return img_b64


# Generate individual chart base64 images optimized for a two-column grid width
img_1 = generate_chart_base64(
    lambda ax: sns.barplot(
        ax=ax,
        data=df_dist,
        x="rating",
        y="review_count",
        hue="rating",
        palette="viridis",
        legend=False,
    )
)
img_2 = generate_chart_base64(
    lambda ax: sns.barplot(
        ax=ax,
        data=df_reviewers,
        x="total_reviews",
        y="username",
        hue="username",
        palette="pastel",
        legend=False,
    )
)
img_3 = generate_chart_base64(
    lambda ax: sns.barplot(
        ax=ax,
        data=df_metrics,
        x="avg_rating",
        y="title",
        hue="title",
        palette="mako",
        legend=False,
    )
)
img_4 = generate_chart_base64(
    lambda ax: sns.barplot(
        ax=ax,
        data=df_popular,
        x="review_count",
        y="title",
        hue="title",
        palette="rocket",
        legend=False,
    )
)
img_5 = generate_chart_base64(
    lambda ax: sns.barplot(
        ax=ax,
        data=df_upvotes,
        x="total_upvotes",
        y="title",
        hue="title",
        palette="magma",
        legend=False,
    )
)
img_6 = generate_chart_base64(
    lambda ax: sns.scatterplot(
        ax=ax,
        data=df_scatter,
        x="review_count",
        y="total_upvotes",
        color="darkorange",
        alpha=0.6,
        s=80,
    )
)
img_7 = generate_chart_base64(
    lambda ax: sns.scatterplot(
        ax=ax,
        data=df_scatter,
        x="review_count",
        y="avg_rating",
        color="teal",
        alpha=0.6,
        s=80,
    )
)
img_8 = generate_chart_base64(
    lambda ax: sns.scatterplot(
        ax=ax,
        data=df_scatter,
        x="total_upvotes",
        y="avg_rating",
        color="purple",
        alpha=0.6,
        s=80,
    )
)

# 4. Assemble Executive Summary & KPIs
total_scraped = f"{df_kpi['total_reviews'][0]:,}"
latest_date = df_kpi["latest_scrape"][0].strftime("%B %d, %Y")

exec_summary = f"""
<strong>Executive Summary: IMDb Corpus & Sentiment Analysis Foundation</strong><br>
Generated on: {datetime.now().strftime('%B %d, %Y')}<br><br>
This automated extraction report audits the foundational dataset powering the IMDb Movie Sentiment Analyzer pipeline. The current batch contains <strong>{total_scraped}</strong> validated reviews across <strong>{df_kpi['total_movies'][0]:,}</strong> unique titles. The database reflects the most recent scrape activity from {latest_date}.
"""

df_kpi_display = df_kpi.rename(
    columns={
        "total_reviews": "Total IMDb Reviews",
        "avg_rating": "Overall Avg Rating",
        "total_movies": "Total Movies Tracked",
        "latest_scrape": "Latest Extraction Timestamp",
    }
)

# 5. Generate HTML Output using a standard template string to completely avoid f-string syntax errors on CSS
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>IMDb NLP Extraction Report</title>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f3f4f6; margin: 0; padding: 40px; color: #1f2937; }}
        .container {{ max-width: 1350px; margin: 0 auto; background: white; padding: 40px; border-radius: 12px; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); }}
        h1 {{ color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 15px; margin-top: 0; }}
        h2 {{ color: #374151; margin-top: 35px; }}
        .summary {{ background-color: #eff6ff; border-left: 5px solid #3b82f6; padding: 20px; border-radius: 0 8px 8px 0; font-size: 16px; line-height: 1.6; }}
        .dataframe {{ width: 100%; border-collapse: collapse; margin-top: 15px; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1); background-color: white; }}
        .dataframe th, .dataframe td {{ padding: 18px; text-align: center; border-bottom: 1px solid #e5e7eb; }}
        .dataframe th {{ background-color: #2563eb; color: white; font-weight: 600; text-transform: uppercase; font-size: 14px; letter-spacing: 0.5px; }}
        .dataframe td {{ font-size: 20px; font-weight: bold; color: #1f2937; }}
        
        .grid-container {{
            display: grid;
            grid-template-columns: repeat(2, 1fr);
            gap: 25px;
            margin-top: 25px;
        }}
        
        .chart-card {{ 
            background: #fff; 
            border: 1px solid #e5e7eb; 
            padding: 20px; 
            border-radius: 8px; 
            box-shadow: 0 2px 4px rgba(0,0,0,0.05);
            display: flex;
            flex-direction: column;
            justify-content: space-between;
        }}
        .chart-card h3 {{ font-size: 16px; margin-top: 0; min-height: 40px; }}
        .chart-card img {{ max-width: 100%; height: auto; border-radius: 4px; display: block; margin: 0 auto; }}
        .chart-commentary {{ margin-top: 15px; padding: 12px; background-color: #fdfbf7; border-left: 4px solid #d97706; font-size: 14px; line-height: 1.5; color: #374151; }}
        
        @media (max-width: 900px) {{
            .grid-container {{ grid-template-columns: 1fr; }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>Automated IMDb NLP Extraction Report</h1>
        
        <div class="summary">
            <p style="margin: 0;">{exec_summary}</p>
        </div>
        
        <h2>KPI Summary Table</h2>
        {kpi_table}
        
        <h2>Visual Insights & Chart-by-Chart Analysis</h2>

        <div class="grid-container">
            <div class="chart-card">
                <div>
                    <h3>Chart 1: Distribution of Valid IMDb Ratings</h3>
                    <img src="data:image/png;base64,{img_1}" alt="Chart 1">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Demonstrates a strong sentiment distribution across the dataset. The majority of user reviews score heavily toward the higher end (peaking at <strong>{top_rating}/10</strong> with <strong>{peak_review_count}</strong> review entries), while lower scores reflect critical dissent.
                </div>
            </div>

            <div class="chart-card">
                <div>
                    <h3>Chart 2: Top 10 Most Active Reviewers</h3>
                    <img src="data:image/png;base64,{img_2}" alt="Chart 2">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Highlights power contributors within the scraped corpus. <strong>{top_reviewer}</strong> leads user engagement with the highest total review volume, establishing a key benchmark for user-level contributions.
                </div>
            </div>

            <div class="chart-card">
                <div>
                    <h3>Chart 3: Top 10 Movies by Average Rating</h3>
                    <img src="data:image/png;base64,{img_3}" alt="Chart 3">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Evaluates top-tier titles maintaining a review volume &gt; 5. <strong>{top_movie_rating}</strong> anchors the upper tier with the highest average sentiment score of the active batch.
                </div>
            </div>

            <div class="chart-card">
                <div>
                    <h3>Chart 4: Top 10 Most Reviewed Movies</h3>
                    <img src="data:image/png;base64,{img_4}" alt="Chart 4">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Identifies high-volume titles driving community discourse. <strong>{top_movie_vol}</strong> stands as the primary anchor for text-heavy data, supplying a rich volume for natural language processing.
                </div>
            </div>

            <div class="chart-card">
                <div>
                    <h3>Chart 5: Top 10 Movies by Helpful Upvotes</h3>
                    <img src="data:image/png;base64,{img_5}" alt="Chart 5">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Tracks community validation via aggregate helpfulness votes. Reviews tied to <strong>{top_movie_upvotes}</strong> have accumulated the highest community visibility and upvote response.
                </div>
            </div>

            <div class="chart-card">
                <div>
                    <h3>Chart 6: Review Volume vs Total Helpful Upvotes</h3>
                    <img src="data:image/png;base64,{img_6}" alt="Chart 6">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Explores structural community interaction: evaluates how the total number of reviews per movie correlates with the cumulative helpfulness upvotes gathered across those reviews.
                </div>
            </div>

            <div class="chart-card">
                <div>
                    <h3>Chart 7: Review Volume vs Average Rating</h3>
                    <img src="data:image/png;base64,{img_7}" alt="Chart 7">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Plots each tracked movie's review count against its average rating (`avg_rating`), showing how user review volume and community scoring distribute across different titles in the database.
                </div>
            </div>

            <div class="chart-card">
                <div>
                    <h3>Chart 8: Total Helpful Upvotes vs Average Rating</h3>
                    <img src="data:image/png;base64,{img_8}" alt="Chart 8">
                </div>
                <div class="chart-commentary">
                    <strong>Analysis:</strong> Cross-examines total helpful upvotes against each movie's average rating (`avg_rating`), illustrating how community engagement levels correspond with positive or negative sentiment scores.
                </div>
            </div>
        </div>

    </div>
</body>
</html>
"""

html_content = html_template.format(
    exec_summary=exec_summary,
    kpi_table=df_kpi_display.to_html(index=False, border=0, classes="dataframe"),
    img_1=img_1,
    img_2=img_2,
    img_3=img_3,
    img_4=img_4,
    img_5=img_5,
    img_6=img_6,
    img_7=img_7,
    img_8=img_8,
    top_rating=top_rating,
    peak_review_count=peak_review_count,
    top_reviewer=top_reviewer,
    top_movie_rating=top_movie_rating,
    top_movie_vol=top_movie_vol,
    top_movie_upvotes=top_movie_upvotes,
)

report_path = output_dir / "extraction_report.html"
report_path.write_text(html_content, encoding="utf-8")

print(
    f"Report successfully saved with precise chart descriptions at '{report_path}'!"
)

Report successfully saved with precise chart descriptions inside the '.' folder!
